# Task 2: Data Preprocessing and Feature Engineering Pipeline
## Fraud Risk Scoring Platform

This notebook demonstrates the end-to-end preprocessing architecture:
1. **Leak-free Data Ingestion & Stratified Train/Test Splitting**
2. **Domain-Driven Feature Engineering** (`is_night_window`, `amount_to_age_ratio`, `declines_risk_interaction`, `log_amount`)
3. **Pipeline Construction with Scikit-Learn `ColumnTransformer`**
4. **Strict `fit` on Training Data & `transform` on Unseen Test Data**
5. **Parity Check with Production REST API Payload Schema**

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.preprocessing import create_preprocessing_pipeline, load_and_prepare_data, FraudFeatureEngineer

## 1. Load Data & Perform Stratified Train/Test Split
- Raw identifier columns (`trans_num`, `cc_num`, `first`, `last`, `street`) are dropped to prevent identity memorization and data leakage.
- Stratified splitting ensures the training and test sets maintain the exact same operational fraud prevalence (~0.39%).

In [2]:
X_train, X_test, y_train, y_test = load_and_prepare_data(
    data_path='../data/raw/fraudTest.csv',
    sample_size=50000,
    test_size=0.2,
    random_state=42
)

print('X_train shape:', X_train.shape)
print('X_test shape: ', X_test.shape)
print('Train Fraud Rate:', f'{y_train.mean()*100:.3f}%')
print('Test Fraud Rate: ', f'{y_test.mean()*100:.3f}%')
X_train.head()

## 2. Inspect Custom Feature Engineering Transformer
The `FraudFeatureEngineer` transformer dynamically creates:
- `is_night_window`: Flags transactions occurring between 22:00 and 04:00 (when fraud spikes by 5x).
- `amount_to_age_ratio`: $\frac{\text{amount}}{\text{customer\_age\_days} + 1}$, capturing rapid spending on new customer accounts.
- `declines_risk_interaction`: $\text{previous\_declines\_24h} \times \ln(1 + \text{amount})$, detecting card-probing attacks.
- `log_amount`: $\ln(1 + \text{amount})$ to compress the heavy right skew of financial spend.

In [3]:
engineer = FraudFeatureEngineer()
X_train_engineered = engineer.transform(X_train)
print('Engineered Columns:', X_train_engineered.columns.tolist())
X_train_engineered[['amount', 'log_amount', 'transaction_hour', 'is_night_window', 'amount_to_age_ratio', 'declines_risk_interaction']].head()

## 3. Construct and Fit Leak-Free Scikit-Learn Pipeline
- Numerical columns: Imputed with `median` and scaled with `RobustScaler` (immune to extreme financial outliers).
- Categorical columns (`merchant_category`): Imputed with `most_frequent` and encoded using `OneHotEncoder(handle_unknown='ignore')` to ensure unseen categories in production do not break inference.

In [4]:
pipeline = create_preprocessing_pipeline()

# Strict rule: Fit ONLY on training data to prevent data leakage
X_train_transformed = pipeline.fit_transform(X_train)
X_test_transformed = pipeline.transform(X_test)

print(f'Transformed X_train matrix shape: {X_train_transformed.shape}')
print(f'Transformed X_test matrix shape:  {X_test_transformed.shape}')

## 4. Test Single API Payload Compatibility
Verifying that raw input matching the REST API specification transforms without errors.

In [5]:
api_payload = {
    'amount': 249.50,
    'merchant_category': 'online',
    'transaction_hour': 2,
    'customer_age_days': 180,
    'previous_declines_24h': 3
}

single_transformed = pipeline.transform(pd.DataFrame([api_payload]))
print('API sample successfully transformed into shape:', single_transformed.shape)
print('Transformed feature vector:\n', single_transformed[0])

## 5. Summary of Preprocessing Decisions

| Decision | Choice | Technical Justification |
| :--- | :--- | :--- |
| **Imputation** | `SimpleImputer(strategy='median')` | Resilient to extreme positive skew in transaction spend amounts. |
| **Scaling** | `RobustScaler` | Uses median and IQR (25th-75th percentile), preventing high-dollar fraud attacks ($1000+) from compressing normal transaction variance. |
| **Encoding** | `OneHotEncoder(handle_unknown='ignore')` | Merchant categories have no natural ordering; `handle_unknown='ignore'` prevents runtime crashes when novel categories arrive via the REST API. |
| **Derived Features** | 4 domain features | Captures nighttime fraud spikes, young-account velocity, card probing declines, and log-scaled amounts. |
| **Imbalance Strategy** | Cost-sensitive learning (`class_weight='balanced'`) | Penalizes false negatives in model optimization without synthesizing artificial transactions or risking test-set contamination. |
| **Data Leakage Safeguard** | Drop IDs, fit train only | Eliminates cardholder memorization and guarantees strict separation of evaluation data. |